# 🐍 Venomous vs Non-Venomous Snake Classification using ResNet50

This notebook demonstrates how to classify snakes as **Venomous** or **Non-Venomous** using Transfer Learning with a pretrained **ResNet50** model.

This project has real-world impact — it can assist **rural communities, forest rangers, and wildlife rescue teams** in quickly identifying dangerous snakes from a photo.

---
### 📋 Workflow
1. Install & Import Libraries
2. Scrape Dataset using `icrawler`
3. Clean & Validate Images
4. Explore & Visualize Dataset
5. Apply Data Augmentation
6. Build Fine-Tuned ResNet50 Model
7. Train with Differential Learning Rates
8. Evaluate — Confusion Matrix & Per-Class Accuracy
9. Grad-CAM Visualisation
10. Predict on New Images

## ⚙️ Step 1 — Install & Import Libraries

In [ ]:
!pip install icrawler torch torchvision pillow matplotlib seaborn scikit-learn -q

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models, datasets
from torchvision.utils import make_grid

# Evaluation
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 📸 Step 2 — Scrape Dataset using icrawler

We scrape images for two classes using targeted Indian species keywords:

| Class | Species targeted |
|---|---|
| **Venomous** | Indian Cobra, Russell's Viper, Krait, King Cobra, Saw-scaled Viper |
| **Non-Venomous** | Rat Snake, Indian Rock Python, Checkered Keelback, Vine Snake (non-venomous) |

> ⏱️ Scraping takes ~5–10 minutes. Skip and use your own dataset if preferred.

In [ ]:
from icrawler.builtin import BingImageCrawler

# ── Keyword sets ─────────────────────────────────────────────────────────────
VENOMOUS_KEYWORDS = [
    'indian cobra naja naja snake',
    'russell viper snake india',
    'common krait bungarus snake india',
    'king cobra ophiophagus snake',
    'saw scaled viper echis india',
]

NON_VENOMOUS_KEYWORDS = [
    'indian rat snake ptyas mucosa',
    'indian rock python python molurus',
    'checkered keelback xenochrophis snake',
    'trinket snake coelognathus helena',
    'buff striped keelback amphiesma snake india',
]

IMAGES_PER_KEYWORD = 60   # adjust up for a larger dataset

def scrape_images(keywords, save_dir, images_per_kw=IMAGES_PER_KEYWORD):
    os.makedirs(save_dir, exist_ok=True)
    for kw in keywords:
        safe_kw = kw.replace(' ', '_')[:30]
        tmp_dir = f'/tmp/crawl_{safe_kw}'
        crawler = BingImageCrawler(storage={'root_dir': tmp_dir})
        crawler.crawl(keyword=kw, max_num=images_per_kw)
        # Move all downloaded files to the class directory
        for fname in os.listdir(tmp_dir):
            src = os.path.join(tmp_dir, fname)
            dst = os.path.join(save_dir, f'{safe_kw}_{fname}')
            shutil.move(src, dst)
        shutil.rmtree(tmp_dir, ignore_errors=True)
        print(f'  ✓ {kw}')

print('Scraping Venomous images...')
scrape_images(VENOMOUS_KEYWORDS,     'data/raw/venomous')
print('\nScraping Non-Venomous images...')
scrape_images(NON_VENOMOUS_KEYWORDS, 'data/raw/non_venomous')
print('\n✅ Scraping complete!')

## 🧹 Step 3 — Clean & Validate Images

Remove:
- Corrupt / unreadable files
- Images smaller than 80×80 pixels
- Non-RGB images (e.g. RGBA PNGs, grayscale)

Then split into `train/` and `val/` (80/20).

In [ ]:
MIN_SIZE = 80   # minimum width and height in pixels

def clean_directory(directory):
    removed, kept = 0, 0
    for fname in os.listdir(directory):
        fpath = os.path.join(directory, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
            with Image.open(fpath) as img:
                w, h = img.size
                if w < MIN_SIZE or h < MIN_SIZE or img.mode != 'RGB':
                    raise ValueError(f'Too small or wrong mode: {img.mode} {w}x{h}')
            kept += 1
        except Exception as e:
            os.remove(fpath)
            removed += 1
    return kept, removed

for cls in ['venomous', 'non_venomous']:
    k, r = clean_directory(f'data/raw/{cls}')
    print(f'{cls}: kept {k}, removed {r}')

In [ ]:
def train_val_split(raw_dir, out_dir, val_ratio=0.2, seed=SEED):
    random.seed(seed)
    for cls in os.listdir(raw_dir):
        cls_path = os.path.join(raw_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        files = os.listdir(cls_path)
        random.shuffle(files)
        split = int(len(files) * (1 - val_ratio))
        splits = {'train': files[:split], 'val': files[split:]}
        for split_name, split_files in splits.items():
            dest = os.path.join(out_dir, split_name, cls)
            os.makedirs(dest, exist_ok=True)
            for f in split_files:
                shutil.copy(os.path.join(cls_path, f), os.path.join(dest, f))
        print(f'{cls}: {split} train | {len(files)-split} val')

train_val_split('data/raw', 'data/split')
print('\n✅ Train/Val split complete!')

## 📊 Step 4 — Explore & Visualize the Dataset

In [ ]:
# Count images per class per split
for split in ['train', 'val']:
    print(f'\n{split.upper()}:')
    for cls in os.listdir(f'data/split/{split}'):
        n = len(os.listdir(f'data/split/{split}/{cls}'))
        print(f'  {cls}: {n} images')

In [ ]:
# ── Figure 1: Class Distribution ─────────────────────────────────────────────
classes = ['venomous', 'non_venomous']
train_counts = [len(os.listdir(f'data/split/train/{c}')) for c in classes]
val_counts   = [len(os.listdir(f'data/split/val/{c}'))   for c in classes]

x = np.arange(len(classes))
fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - 0.2, train_counts, 0.35, label='Train', color='steelblue')
bars2 = ax.bar(x + 0.2, val_counts,   0.35, label='Val',   color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(['Venomous 🐍', 'Non-Venomous 🟢'], fontsize=12)
ax.set_ylabel('Image Count')
ax.set_title('Dataset Class Distribution', fontsize=13, fontweight='bold')
ax.legend()
ax.bar_label(bars1, padding=3)
ax.bar_label(bars2, padding=3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Sample images from each class ───────────────────────────────────
def show_samples(class_dir, class_name, n=5, ax_row=None):
    files = random.sample(os.listdir(class_dir), min(n, len(os.listdir(class_dir))))
    for i, fname in enumerate(files):
        img = Image.open(os.path.join(class_dir, fname)).convert('RGB')
        img = img.resize((150, 150))
        ax_row[i].imshow(img)
        ax_row[i].axis('off')
        if i == 0:
            ax_row[i].set_ylabel(class_name, fontsize=11, fontweight='bold', rotation=0,
                                 labelpad=60, va='center')

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
show_samples('data/split/train/venomous',     '🐍 Venomous',     5, axes[0])
show_samples('data/split/train/non_venomous', '🟢 Non-Venomous', 5, axes[1])
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔄 Step 5 — Data Augmentation & DataLoaders

Training transforms include aggressive augmentation to help the model generalise across:
- Different lighting conditions (forest, daylight, flash)
- Various orientations (snakes coil in any direction)
- Partial occlusion by grass/rocks

Validation uses only Resize + Normalize — no augmentation.

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32

# ImageNet mean/std for ResNet normalisation
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),   # simulate occlusion
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = datasets.ImageFolder('data/split/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder('data/split/val',   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

CLASS_NAMES = train_dataset.classes
print(f'Classes  : {CLASS_NAMES}')
print(f'Train    : {len(train_dataset)} images')
print(f'Val      : {len(val_dataset)} images')

In [ ]:
# ── Figure 3: Augmentation examples (same image, 8 transforms) ───────────────
sample_path = random.choice(
    [os.path.join('data/split/train/venomous', f)
     for f in os.listdir('data/split/train/venomous')])
orig_img = Image.open(sample_path).convert('RGB')

aug_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomPerspective(0.3, p=0.5),
])

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes[0][0].imshow(orig_img.resize((224, 224)))
axes[0][0].set_title('Original', fontweight='bold')
axes[0][0].axis('off')

all_axes = [axes[r][c] for r in range(2) for c in range(5)][1:]
for ax in all_axes:
    augmented = aug_transform(orig_img)
    ax.imshow(augmented)
    ax.axis('off')

plt.suptitle('Data Augmentation Examples (same source image)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🧠 Step 6 — Build Fine-Tuned ResNet50 Model

Architecture:
```
ResNet50 backbone (pretrained on ImageNet)
  → Freeze all layers except layer4
  → Replace FC head:
       Linear(2048 → 512) → BatchNorm → ReLU → Dropout(0.4)
       → Linear(512 → 128) → ReLU → Dropout(0.3)
       → Linear(128 → 2)
```

> A deeper custom head than the cattle/buffalo project — snake inter-class similarity is higher, so the classifier needs more capacity.

In [ ]:
def build_snake_classifier(num_classes=2, dropout=0.4):
    # Load pretrained ResNet50
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze layer3 and layer4 for fine-tuning
    for param in model.layer3.parameters():
        param.requires_grad = True
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Replace classification head
    in_features = model.fc.in_features  # 2048
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_classes)
    )
    return model

model = build_snake_classifier().to(DEVICE)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

## 🏋️ Step 7 — Train with Differential Learning Rates

Different parts of the network get different learning rates:
- `layer3` (deep features): `5e-5` — very careful fine-tuning
- `layer4` (high-level features): `1e-4`
- Custom head: `1e-3` — learns fast from scratch

In [ ]:
EPOCHS = 25

# Differential learning rates
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 5e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 1e-3},
], weight_decay=1e-4)

scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)
criterion  = nn.CrossEntropyLoss()

# ── Training loop ─────────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # ── Validate ──
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    # ── Metrics ──
    t_loss = train_loss / len(train_dataset)
    v_loss = val_loss   / len(val_dataset)
    t_acc  = 100 * train_correct / len(train_dataset)
    v_acc  = 100 * val_correct   / len(val_dataset)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    scheduler.step(v_loss)

    # Save best model
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), 'best_snake_model.pth')

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {t_loss:.4f} Acc: {t_acc:.1f}% | '
          f'Val Loss: {v_loss:.4f} Acc: {v_acc:.1f}%')

print(f'\n✅ Best Val Accuracy: {best_val_acc:.1f}%')

In [ ]:
# ── Figure 4: Training Curves ─────────────────────────────────────────────────
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', markersize=4, label='Val Loss')
axes[0].set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', markersize=4, label='Val Acc')
axes[1].set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 📈 Step 8 — Evaluate — Confusion Matrix, Per-Class Accuracy & ROC Curve

In [ ]:
# Load best saved model
model.load_state_dict(torch.load('best_snake_model.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        preds   = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy()[:, 1])  # prob of venomous

print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
print(f'ROC-AUC Score: {roc_auc_score(all_labels, all_probs):.4f}')

In [ ]:
# ── Figure 5: Confusion Matrix + ROC Curve ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score   = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC AUC = {auc_score:.3f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🔥 Step 9 — Grad-CAM Visualisation

**Grad-CAM** (Gradient-weighted Class Activation Mapping) shows *where* in the image the model is looking to make its decision.

This is especially important for snake classification — we want to verify the model is looking at the **snake's body/head features** (scale patterns, head shape) and NOT background elements like grass or rocks.

> This is a unique addition over standard classification projects — it adds model interpretability.

In [ ]:
import torch.nn.functional as F
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def fwd_hook(module, input, output):
            self.activations = output.detach()
        def bwd_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(fwd_hook)
        self.target_layer.register_backward_hook(bwd_hook)

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


def overlay_gradcam(img_path, model, gradcam):
    img = Image.open(img_path).convert('RGB').resize((224, 224))
    tensor = val_transforms(img).unsqueeze(0).to(DEVICE)
    cam, pred_idx = gradcam.generate(tensor)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = np.array(img) * 0.6 + heatmap * 0.4
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return img, overlay, CLASS_NAMES[pred_idx]


gradcam = GradCAM(model, model.layer4[-1])

# ── Figure 6: Grad-CAM on 4 validation images ────────────────────────────────
sample_paths = []
for cls in CLASS_NAMES:
    cls_dir = f'data/split/val/{cls}'
    files   = os.listdir(cls_dir)
    sample_paths += [os.path.join(cls_dir, f) for f in random.sample(files, 2)]

fig, axes = plt.subplots(4, 2, figsize=(10, 16))
for i, path in enumerate(sample_paths):
    orig, overlay, pred = overlay_gradcam(path, model, gradcam)
    true_cls = Path(path).parent.name
    axes[i][0].imshow(orig)
    axes[i][0].set_title(f'Original\nTrue: {true_cls}', fontsize=10)
    axes[i][0].axis('off')
    axes[i][1].imshow(overlay)
    axes[i][1].set_title(f'Grad-CAM\nPred: {pred}', fontsize=10,
                         color='green' if pred == true_cls else 'red')
    axes[i][1].axis('off')

plt.suptitle('Grad-CAM: What the Model Looks At', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔬 Step 10 — Predict on New Images

Upload any snake image and get an instant prediction with confidence score and Grad-CAM overlay.

In [ ]:
def predict_snake(image_path: str, model=model, show_gradcam: bool = True):
    """
    Predict whether a snake is Venomous or Non-Venomous.

    Parameters
    ----------
    image_path  : str  — path to the input image
    show_gradcam: bool — whether to display Grad-CAM overlay
    """
    model.load_state_dict(torch.load('best_snake_model.pth', map_location=DEVICE))
    model.eval()

    img    = Image.open(image_path).convert('RGB')
    tensor = val_transforms(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0]
        pred   = probs.argmax().item()

    label      = CLASS_NAMES[pred]
    confidence = probs[pred].item() * 100
    danger     = '🔴 VENOMOUS — Exercise Caution!' if label == 'venomous' else '🟢 Non-Venomous — Generally Safe'

    print(f'Prediction  : {label.upper()}')
    print(f'Confidence  : {confidence:.1f}%')
    print(f'Assessment  : {danger}')
    print(f'\nClass Probabilities:')
    for i, cls in enumerate(CLASS_NAMES):
        bar = '█' * int(probs[i].item() * 30)
        print(f'  {cls:<15} {bar} {probs[i].item()*100:.1f}%')

    if show_gradcam:
        gc    = GradCAM(model, model.layer4[-1])
        _, overlay, _ = overlay_gradcam(image_path, model, gc)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(img.resize((224, 224)))
        axes[0].set_title('Input Image', fontweight='bold')
        axes[0].axis('off')
        axes[1].imshow(overlay)
        axes[1].set_title(f'Grad-CAM | Pred: {label} ({confidence:.1f}%)', fontweight='bold',
                          color='red' if label == 'venomous' else 'green')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

    return label, confidence


# ── Example usage ─────────────────────────────────────────────────────────────
# predict_snake('path/to/your/snake_image.jpg')

# Demo on a validation image
demo_img = random.choice(
    [os.path.join('data/split/val/venomous', f)
     for f in os.listdir('data/split/val/venomous')])
predict_snake(demo_img)

---
## 📋 Results Summary

| Class | Precision | Recall | F1-Score | Accuracy |
|---|---|---|---|---|
| Venomous | ~0.90 | ~0.91 | ~0.90 | ~90% |
| Non-Venomous | ~0.91 | ~0.90 | ~0.90 | ~90% |
| **Overall** | | | | **~90%** |

> Actual results depend on dataset size and quality.

---
## 🔄 Comparison: Cattle/Buffalo vs Snake Classification

| Aspect | Cattle vs Buffalo | Snake Classification |
|---|---|---|
| Inter-class similarity | Low | **High** (scales, color similar) |
| Fine-tuned layers | layer4 only | **layer3 + layer4** |
| Custom head depth | 2 layers | **3 layers** |
| Extra evaluation | Confusion Matrix | **+ ROC-AUC Curve** |
| Interpretability | None | **Grad-CAM** |
| Real-world impact | Agricultural | **Safety-critical** |

---
## 🚀 Future Improvements

1. **Multi-class** — Extend to species-level classification (cobra vs krait vs viper)
2. **EfficientNetV2** — Benchmark against ResNet50 for accuracy vs speed
3. **Mobile deployment** — Convert to TFLite/ONNX for field use on ranger smartphones
4. **Uncertainty estimation** — Add Monte Carlo Dropout so the model flags low-confidence predictions